In [ ]:
# Dependencies

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.impute import SimpleImputer
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score, 
                             precision_recall_curve, roc_curve, f1_score, precision_score, recall_score)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import re
import os
warnings.filterwarnings('ignore')

# STEP 1 — Load and clean the raw table

In [ ]:
# ======================================================================
# STEP 1: LOAD AND PREPARE DATA (REFERENCE + MULTI-KIT MONORAIL)
# ======================================================================

def load_data(model_path, monorail_paths):
    """
    Load and prepare reference (model) data and Monorail data (one or more kits),
    align common columns, and return a single combined DataFrame.

    Parameters
    ----------
    model_path : str
        Path to model.csv (reference experimental campaign).
    monorail_paths : str or list of str
        Path or list of paths to Monorail TestBrakefinal_data_kitXX.csv files.

    Returns
    -------
    df_base : pandas.DataFrame
        Combined DataFrame with:
        - aligned common columns between reference and Monorail,
        - binary label (0/1) where available,
        - 'Source' column (kit ID or 0 for reference),
        - 'DataSource' column (0 = reference, 1 = Monorail).
    """

    # --------------------------------------------------------------
    # Helper: load and clean ONE Monorail file
    # --------------------------------------------------------------
    def load_Monorail(filepath: str) -> pd.DataFrame:
        df = pd.read_csv(filepath)

        # Keep only standard braking
        if 'Non_Standard_Braking' in df.columns:
            df = df[df['Non_Standard_Braking'] == 0]

        # Extract numeric kit ID from filename, e.g. "TestBrakefinal_data_kit06.csv" -> 6
        match = re.search(r'Dati(\d+)', os.path.basename(filepath))
        source = int(match.group(1)) if match else -1
        df['Source'] = source
        df['Malfunction'] = 0

        # Convert "xx sec" string columns to float seconds where possible
        for col in df.select_dtypes(include='object'):
            try:
                df[col] = df[col].str.replace(' sec', '', regex=False).astype(float)
            except (AttributeError, ValueError):
                # AttributeError if column is not string-like; ValueError if some values cannot be cast
                continue

        return df

    # --------------------------------------------------------------
    # 1) REFERENCE DATA: load, label, aggregate
    # --------------------------------------------------------------
    df_reference = pd.read_csv(model_path)
    df_reference['Malfunction'] = df_reference['Malfunction'].astype(str)

    # Binary label from malfunction code
    leakage_codes = ['C', 'D', 'E', 'F', 'G']
    df_reference['LeakageLabel'] = np.where(
        df_reference['Malfunction'].isin(leakage_codes),
        'Combined leakage',
        'Healthy'
    )

    # Add Source = 0 for reference campaign
    df_reference['Source'] = 0

    # Aggregate delay and efficiency columns
    delay_eff_map = {
        'Total_timing_delay':      ['Brake_timing_delay_exp',      'Release_timing_delay_exp'],
        'Total_energy_delay':      ['Brake_energy_delay_exp',      'Release_energy_delay_exp'],
        'Total_power_delay':       ['Brake_power_delay_exp',       'Release_power_delay_exp'],
        'Total_power_efficiency':  ['Brake_power_efficiency_exp',  'Release_power_efficiency_exp'],
        'Total_energy_efficiency': ['Brake_energy_effiency_exp',   'Release_energy_efficiency_exp']
    }

    for new_col, (c1, c2) in delay_eff_map.items():
        # If any of these columns are missing in some version of model.csv, guard with .get
        if c1 in df_reference.columns and c2 in df_reference.columns:
            df_reference[new_col] = df_reference[c1] + df_reference[c2]

    # Drop original per-phase columns (only those that actually exist)
    cols_to_drop = [c for pair in delay_eff_map.values() for c in pair if c in df_reference.columns]
    df_reference.drop(columns=cols_to_drop, inplace=True, errors='ignore')

    # Rename to your canonical names
    rename_map = {
        'Release_start_pressure_delay_exp': 'Release_start_pressure_delay',
        'Buildup_end_pressure_delay_exp':  'Buildup_end_pressure_delay',
        'Weight':                          'WV_MeanPressure',
        'Brake_action':                    'EmergencyBrake_action'
    }
    df_reference.rename(columns=rename_map, inplace=True)

    # --------------------------------------------------------------
    # 2) MONORAIL DATA: load one or more kit files
    # --------------------------------------------------------------
    if isinstance(monorail_paths, str):
        monorail_paths = [monorail_paths]

    dfs_mono = [load_Monorail(fp) for fp in monorail_paths]
    df_data = pd.concat(dfs_mono, ignore_index=True)

    # --------------------------------------------------------------
    # 3) ALIGN STRUCTURES AND COMBINE
    # --------------------------------------------------------------
    # Ensure 'Source' is integer in both
    df_reference['Source'] = df_reference['Source'].astype(int)
    df_data['Source']      = df_data['Source'].astype(int)

    # Columns common to BOTH datasets
    common_cols = df_reference.columns.intersection(df_data.columns).tolist()

    # Subsets with only common columns + a DataSource flag
    df_reference_subset = df_reference[common_cols].copy()
    df_reference_subset['DataSource'] = 0  # 0 = reference campaign

    df_data_subset = df_data[common_cols].copy()
    df_data_subset['DataSource'] = 1       # 1 = Monorail (real-time) data

    # Stack reference + Monorail
    df_combined = pd.concat([df_reference_subset, df_data_subset], ignore_index=True)

    # Encode final label column (will be NaN for Monorail if it has no LeakageLabel)
    if 'LeakageLabel' in df_combined.columns:
        df_combined.rename(columns={'LeakageLabel': 'label'}, inplace=True)
        df_combined['label'] = df_combined['label'].map({'Healthy': 0, 'Combined leakage': 1})

    # Convert any remaining "xx sec" string columns to float (esp. from model.csv)
    for col in df_combined.select_dtypes(include='object'):
        try:
            df_combined[col] = df_combined[col].str.replace(' sec', '', regex=False).astype(float)
        except (AttributeError, ValueError):
            continue
    df_combined["WV_bin"] = df_combined["WV_MeanPressure"].apply(
    lambda p: np.nan if pd.isna(p) else (0 if p < 2 else (2 if p > 3 else 1))
    )
    df_base = df_combined.copy()
    return df_base, df_reference, df_data_subset

model_path = 'model.csv'
monorail_paths = [
    'TestBrakefinal_data_Dati01.csv',
    'TestBrakefinal_data_Dati06.csv',
    'TestBrakefinal_data_Dati27.csv'
]

[df, df_reference, df_monorail] = load_data(model_path, monorail_paths)
df['Malfunction'] = df['Malfunction'].astype(str)
print(df.shape)
print(df['DataSource'].value_counts(dropna=False))
print(df['label'].value_counts(dropna=False))  # will include NaN for unlabeled Monorail

# Cluster one selected features

In [ ]:
# Select the Loading Condition that is within 2 and 3 bar
# Instead of directly filtering, lets create WV_bin column for <2 bar, 2 to 3 bar, and >3 bar 
# based on WV_MeanPressure

from sklearn.model_selection import train_test_split

def preprocess_data(df, features, test_size,
                    label_col="label",
                    mal_col="Malfunction"):
    """
    Preprocess data:
      - filter WV_bin == 1
      - select features
      - split into train/test
      - keep both:
          * binary target (label_col)
          * malfunction codes (mal_col)
      - extract healthy-only training samples

    Parameters
    ----------
    df : pandas.DataFrame
        Full dataset.
    features : list of str
        Feature column names to use.
    test_size : float
        Proportion for test split.
    label_col : str, default 'label'
        Column name of the binary target (0/1).
    mal_col : str, default 'Malfunction'
        Column name of the malfunction code.

    Returns
    -------
    X_train, X_test : pandas.DataFrame
    y_train, y_test : pandas.Series
        Binary label (0/1).
    mal_train, mal_test : pandas.Series
        Malfunction codes (A,B,C,...,0) aligned to X_train/X_test.
    X_train_healthy : pandas.DataFrame
        Subset of X_train where y_train == 0.
    """

    df = df.copy()

    # Filter by WV_bin == 1 (pressure between 2 and 3 bar)
    df_filt = df[df["WV_bin"] == 1].copy()

    # Validate feature selection
    missing = [f for f in features if f not in df_filt.columns]
    if missing:
        raise ValueError(f"The following features are not in the dataframe: {missing}")

    # Features and targets
    X      = df_filt[features]
    y      = df_filt[label_col]       # main binary target
    y_mal  = df_filt[mal_col]         # malfunction codes

    # Joint split so everything stays aligned
    X_train, X_test, y_train, y_test, mal_train, mal_test = train_test_split(
        X, y, y_mal,
        test_size=test_size,
        stratify=y,
        random_state=42
    )

    # Healthy training samples
    X_train_healthy = X_train[y_train == 0]

    print(f"Total samples: {len(df)}")
    print(f"Filtered samples (WV_bin==1): {len(df_filt)}")
    print(f"Training samples: {len(X_train)} (Healthy: {sum(y_train==0)}, Leakage: {sum(y_train==1)})")
    print(f"Training samples (healthy only): {len(X_train_healthy)}")
    print(f"Test samples: {len(X_test)} (Healthy: {sum(y_test==0)}, Leakage: {sum(y_test==1)})")

    return (
        X_train, X_test,
        y_train, y_test,
        mal_train, mal_test,
        X_train_healthy
    )


In [ ]:
# ============================================================================
# STEP 1: DATA PREPROCESSING TRAIN TEST SPLIT AND FEATURE SELECTION 
# ============================================================================
selected_features = ['Total_power_efficiency'] 

[X_train, X_test, y_train, y_test, mal_train, mal_test, X_train_healthy] = preprocess_data(df, selected_features, test_size=0.2)
X_train.head()

In [ ]:
from sklearn.cluster import KMeans
X = X_train.values

kmeans = KMeans(n_clusters=5, n_init=10)
labels_km = kmeans.fit_predict(X.reshape(-1, 1))


In [ ]:
from sklearn.mixture import GaussianMixture

gmm = GaussianMixture(n_components=5, covariance_type="full", random_state=0)
labels_gmm = gmm.fit_predict(X.reshape(-1, 1))
proba  = gmm.predict_proba(X.reshape(-1, 1))


In [ ]:
from sklearn.metrics import silhouette_score

sil_km  = silhouette_score(X.reshape(-1, 1), labels_km)
sil_gmm = silhouette_score(X.reshape(-1, 1), labels_gmm)

print("Silhouette KMeans:", sil_km)
print("Silhouette GMM:", sil_gmm)


In [ ]:
print("GMM log-likelihood:", gmm.score(X.reshape(-1, 1)))
print("GMM AIC:", gmm.aic(X.reshape(-1, 1)))
print("GMM BIC:", gmm.bic(X.reshape(-1, 1)))


In [ ]:
import numpy as np

def cluster_stability_kmeans(X, K, n_runs=20):
    labels_all = []
    for i in range(n_runs):
        km = KMeans(n_clusters=K, n_init=1)
        labels_all.append(km.fit_predict(X.reshape(-1,1)))
    return np.mean([
        np.mean(labels_all[0] == labels_all[i])
        for i in range(1, n_runs)
    ])

def cluster_stability_gmm(X, K, n_runs=20):
    labels_all = []
    for i in range(n_runs):
        gm = GaussianMixture(n_components=K, random_state=i)
        labels_all.append(gm.fit_predict(X.reshape(-1,1)))
    return np.mean([
        np.mean(labels_all[0] == labels_all[i])
        for i in range(1, n_runs)
    ])

print("KMeans stability:", cluster_stability_kmeans(X, K))
print("GMM stability:", cluster_stability_gmm(X, K))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

x = X.flatten()

kde = gaussian_kde(x)
xx = np.linspace(x.min(), x.max(), 1000)
pdf = kde(xx)

plt.figure(figsize=(7,4))
plt.hist(x, bins=40, density=True, alpha=0.4, label="Data")
plt.plot(xx, pdf, 'k', lw=2, label="KDE")
plt.xlabel("Feature value")
plt.ylabel("Density")
plt.legend()
plt.title("1D Feature Distribution with KDE")
plt.show()


In [ ]:
from scipy.signal import find_peaks

peaks, props = find_peaks(pdf, prominence=0.01)
print("Suggested K from KDE peaks:", len(peaks))

plt.figure(figsize=(7,4))
plt.plot(xx, pdf, 'k')
plt.plot(xx[peaks], pdf[peaks], 'ro')
plt.title("Detected density peaks")
plt.show()


In [ ]:
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
idx = np.argsort(x)
x_sorted = x[idx]

K = 5  # try 2,3,4

km = KMeans(n_clusters=K, n_init=20, random_state=0)
labels_km = km.fit_predict(x.reshape(-1,1))[idx]

gmm = GaussianMixture(n_components=K, random_state=0)
labels_gmm = gmm.fit_predict(x.reshape(-1,1))[idx]


In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score

x = X_train.values.astype(float).reshape(-1, 1)

Ks = range(2, 8)  # silhouette needs K>=2
sil_km, sil_gmm = [], []

for k in Ks:
    km = KMeans(n_clusters=k, n_init=20, random_state=0)
    lab_km = km.fit_predict(x)
    sil_km.append(silhouette_score(x, lab_km))

    gm = GaussianMixture(n_components=k, random_state=0)
    lab_gm = gm.fit_predict(x)
    sil_gmm.append(silhouette_score(x, lab_gm))

print("Best K (KMeans) by silhouette:", Ks[int(np.argmax(sil_km))])
print("Best K (GMM) by silhouette:", Ks[int(np.argmax(sil_gmm))])


In [ ]:
aic, bic = [], []
Ks = range(1, 10)

for k in Ks:
    gm = GaussianMixture(n_components=k, random_state=0)
    gm.fit(x)
    aic.append(gm.aic(x))
    bic.append(gm.bic(x))

best_k_bic = Ks[int(np.argmin(bic))]
best_k_aic = Ks[int(np.argmin(aic))]

print("Best K by BIC:", best_k_bic)
print("Best K by AIC:", best_k_aic)


In [ ]:
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score

Ks = range(2, 10)
ch_km, db_km = [], []
ch_gm, db_gm = [], []

for k in Ks:
    km = KMeans(n_clusters=k, n_init=20, random_state=0)
    lab_km = km.fit_predict(x)
    ch_km.append(calinski_harabasz_score(x, lab_km))
    db_km.append(davies_bouldin_score(x, lab_km))

    gm = GaussianMixture(n_components=k, random_state=0)
    lab_gm = gm.fit_predict(x)
    ch_gm.append(calinski_harabasz_score(x, lab_gm))
    db_gm.append(davies_bouldin_score(x, lab_gm))

print("Best KMeans K by CH:", Ks[int(np.argmax(ch_km))])
print("Best KMeans K by DB:", Ks[int(np.argmin(db_km))])
print("Best GMM K by CH:", Ks[int(np.argmax(ch_gm))])
print("Best GMM K by DB:", Ks[int(np.argmin(db_gm))])


In [ ]:
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score

Ks = range(1, 9)

# --- KMeans inertia ---
inertia = []
for k in Ks:
    km = KMeans(n_clusters=k, n_init=20, random_state=0)
    km.fit(x)
    inertia.append(km.inertia_)

# --- Silhouette (K >= 2) ---
Ks_sil = range(2, 9)
sil_km, sil_gmm = [], []

for k in Ks_sil:
    km = KMeans(n_clusters=k, n_init=20, random_state=0)
    lab_km = km.fit_predict(x)
    sil_km.append(silhouette_score(x, lab_km))

    gm = GaussianMixture(n_components=k, random_state=0)
    lab_gm = gm.fit_predict(x)
    sil_gmm.append(silhouette_score(x, lab_gm))

# --- GMM AIC / BIC ---
aic, bic = [], []
for k in Ks:
    gm = GaussianMixture(n_components=k, random_state=0)
    gm.fit(x)
    aic.append(gm.aic(x))
    bic.append(gm.bic(x))


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 1, figsize=(7, 10), sharex=True)

# --- (1) KMeans Elbow ---
axes[0].plot(Ks, inertia, 'o-')
axes[0].set_ylabel("Inertia")
axes[0].set_title("KMeans – Elbow method")
axes[0].grid(True)

# --- (2) Silhouette ---
axes[1].plot(Ks_sil, sil_km, 'o-', label="KMeans")
axes[1].plot(Ks_sil, sil_gmm, 's--', label="GMM")
axes[1].set_ylabel("Silhouette score")
axes[1].set_title("Silhouette score vs K")
axes[1].legend()
axes[1].grid(True)

# --- (3) GMM AIC / BIC ---
axes[2].plot(Ks, aic, 'o-', label="AIC")
axes[2].plot(Ks, bic, 's-', label="BIC")
axes[2].set_xlabel("Number of clusters K")
axes[2].set_ylabel("Criterion value")
axes[2].set_title("GMM model order selection")
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.mixture import GaussianMixture

# ---- 1D feature vector (from your X_train) ----
x = X_train.values.astype(float).reshape(-1, 1).ravel()

# ---- Search range for K ----
Ks = range(1, 9)

bic = []
models = []
for k in Ks:
    gm = GaussianMixture(n_components=k, random_state=0)
    gm.fit(x.reshape(-1, 1))
    bic.append(gm.bic(x.reshape(-1, 1)))
    models.append(gm)

best_idx = int(np.argmin(bic))
K_best = Ks[best_idx]
best_gmm = models[best_idx]

print("Selected K by BIC:", K_best)


plt.figure(figsize=(6,4))
plt.plot(list(Ks), bic, 'o-')
plt.xlabel("K (# components)")
plt.ylabel("BIC (lower is better)")
plt.title("GMM model order selection (BIC)")
plt.grid(True)
plt.show()


In [ ]:
labels = best_gmm.predict(x.reshape(-1, 1))  # 0..K_best-1

means = best_gmm.means_.ravel()
order = np.argsort(means)
remap = {old:new for new, old in enumerate(order)}
labels_ord = np.array([remap[l] for l in labels])


In [ ]:
dfp = pd.DataFrame({
    "feature": x,
    "cluster": labels_ord
})

groups = [dfp.loc[dfp["cluster"]==c, "feature"].values for c in sorted(dfp["cluster"].unique())]
names  = [f"C{c} (n={len(groups[c])})" for c in range(len(groups))]

plt.figure(figsize=(8,4))
plt.boxplot(groups, labels=names, showfliers=True)
plt.ylabel("Feature value")
plt.title(f"Feature distribution by GMM clusters (K={K_best})")
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


# STEP 3 — Unsupervised anomaly detection (One-Class SVM)

In [ ]:
ocsvm = OneClassSVM(kernel='rbf', gamma='scale', nu=0.02)
ocsvm.fit(X)

# Predict anomaly flag & score
df_numeric['AnomalyFlag'] = (ocsvm.predict(X) == -1).astype(int)
df_numeric['AnomalyScore'] = -ocsvm.decision_function(X)

print(df_numeric['AnomalyFlag'].value_counts())

In [ ]:
unique, counts = np.unique(df_numeric['Release_timing_pipe'], return_counts=True)
print(unique)

# STEP 4 — Extract anomaly subset

In [ ]:
anom = df_numeric[df_numeric['AnomalyFlag'] == 1].copy()
X_anom = anom.drop(columns=['AnomalyFlag', 'AnomalyScore']).values
print(f"Total anomalies detected: {len(anom)}")



# STEP 5 — Dimensionality reduction (for clustering & plotting)

In [ ]:
# Before PCA, add this:

# Center and scale once again before PCA (safety)
from sklearn.preprocessing import RobustScaler

scaler2 = RobustScaler()
X_scaled = scaler2.fit_transform(X_anom)

pca = PCA(n_components=3, whiten=True, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# Option B (optional): nonlinear UMAP
# reducer = umap.UMAP(n_neighbors=10, min_dist=0.3, random_state=42)
# X_pca = reducer.fit_transform(X_anom)

anom['PC1'], anom['PC2'], anom['PC3'] = X_pca[:,0], X_pca[:,1], X_pca[:,2]


In [ ]:
from sklearn.neighbors import NearestNeighbors
import numpy as np
import matplotlib.pyplot as plt

nn = NearestNeighbors(n_neighbors=5)
nn.fit(X_pca)
distances, _ = nn.kneighbors(X_pca)
distances = np.sort(distances[:, 4])

plt.plot(distances)
plt.ylabel("5th nearest-neighbor distance")
plt.xlabel("Points sorted by distance")
plt.title("DBSCAN epsilon tuning")
plt.show()

# STEP 6 — Cluster anomalies

In [ ]:
selected_features = [
    'Total_power_delay',
    'Buildup_end_pressure_delay',
    'Release_start_pressure_delay',
    'BC_MaxPressure',
    'First_phase_max_gradient_cyl',
    'Energy_ratio',
    'Release_timing_pipe',
    'Total_power_efficiency'
]

In [ ]:
# 1️⃣ Identify which rows are anomalies in the original dataset
anom_mask = df_numeric['AnomalyFlag'] == 1
anom_idx = df_numeric.index[anom_mask]

# 2️⃣ Run clustering on the scaled matrix of those same rows
X_anom = X[anom_mask]          # exact rows used
X_scaled = StandardScaler().fit_transform(X_anom)

pca = PCA(n_components=3, whiten=True, random_state=42)
X_pca = pca.fit_transform(X_scaled)
anom['PC1'], anom['PC2'], anom['PC3'] = X_pca[:,0], X_pca[:,1], X_pca[:,2]

# 3️⃣ Cluster (DBSCAN, KMeans, etc.)
db = DBSCAN(eps=1.4, min_samples=5)
labels = db.fit_predict(X_pca)

# 4️⃣ Reattach labels to the raw data safely
df_clustered = df_numeric.loc[anom_idx].copy()
df_clustered['Cluster'] = labels

# 5️⃣ Summarize using unscaled, physical features
summary = df_clustered.groupby('Cluster')[selected_features].mean()
print(summary)


# STEP 7 — Visualize clusters

In [ ]:
sns.scatterplot(data=anom, x='PC1', y='PC2',
                hue='Cluster', palette='tab10', s=60)
plt.title("Clusters of anomalies (DBSCAN on PCA space)")
plt.show()



# STEP 8 — Compute cluster feature means

In [ ]:
print(df_numeric['Release_timing_pipe'].max())         # expected <= 200
print(anom['Release_timing_pipe'].max())               # should match above
print(anom[['Release_timing_pipe','Cluster']].head())  # confirm numeric range


In [ ]:
selected_features = [
    'Total_power_delay',
    'Buildup_end_pressure_delay',
    'Release_start_pressure_delay',
    'BC_MaxPressure',
    'First_phase_max_gradient_cyl',
    'Energy_ratio',
    'Release_timing_pipe',
    'Total_Power_eff'
]

print(main[selected_features].mean())
print(outliers[selected_features].mean())

In [ ]:
main = anom[anom['Cluster'] == 0]
outliers = anom[anom['Cluster'] == -1]

print(main[selected_features].mean())
print(outliers[selected_features].mean())


In [ ]:
cluster_summary = anom.groupby('Cluster').mean().T
cluster_summary = cluster_summary.sort_index()
sns.heatmap(cluster_summary, cmap='coolwarm', annot=False)
plt.title("Average feature values per cluster (anomalies only)")
plt.show()

print(cluster_summary)



# STEP 9 — Basic heuristic interpretation

In [ ]:
def interpret_cluster(row):
    # You can customize these based on physical meaning
    if row['BC_MaxPressure'] < 0.8 and row['Buildup_gradient_pipe'] < 0.05:
        return 'Probable Leakage'
    elif row['Release_start_pressure_delay'] > 0.1:
        return 'Slow Release / Sticky Valve'
    elif row['Total_power_delay'] > 0.1:
        return 'Energy Delay / Low Efficiency'
    else:
        return 'Normal-like'

summary_T = cluster_summary.T
summary_T['Interpretation'] = summary_T.apply(interpret_cluster, axis=1)
print("\n=== Cluster interpretation ===")
print(summary_T[['Interpretation']])



In [ ]:
# ===============================================================
# STEP 10 — Save outputs
# ===============================================================
anom.to_csv("TestBrake_anomalies_clustered.csv", index=False)
summary_T.to_csv("Cluster_summary.csv")
print("✅ Results saved to CSV files.")